In [1]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
     

import torchvision
import torchvision.transforms as transforms

from models import *
use_gpu = torch.cuda.is_available()
batch_size = 128
model_name = "vgg16_quant_unstruct"
model = VGG16_quant()
use_gpu = torch.cuda.is_available()
device = torch.device("cuda" if use_gpu else "cpu") 
normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])

In [2]:
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))

trainloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))


testloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

In [3]:
print_freq = 390 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)

        input = input.cuda(non_blocking=True)
        target = target.cuda(non_blocking=True)

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input = input.cuda(non_blocking=True)
            target = target.cuda(non_blocking=True)

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

        

In [4]:
import os
import glob
import shutil
import torch

def save_checkpoint(state, is_best, fdir, best_prec):
    os.makedirs(fdir, exist_ok=True)
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)

    if is_best:
        # Remove previous best checkpoint
        for old_file in glob.glob(os.path.join(fdir, 'model_best_prec=*.pth.tar')):
            os.remove(old_file)

        # Save new best checkpoint with accuracy in filename
        best_path = os.path.join(fdir, f'model_best_prec={best_prec:.3f}.pth.tar')
        shutil.copyfile(filepath, best_path)


In [15]:
model = VGG16_quant().cuda()
fdir = f"final_model/final_model_checkpoints/"
checkpoint = torch.load(
    os.path.join(fdir, 'model_best.pth.tar'),
    map_location='cuda',
    weights_only=False
)
device = torch.device("cuda") 
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(),momentum=0.9)
model.load_state_dict(checkpoint['state_dict'])
optimizer.load_state_dict(checkpoint['optimizer'])

model.eval()
model.cuda()

correct = 0
test_loss = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)
print('\n Model Verified Test Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


 Model Verified Test Accuracy: 9254/10000 (93%)



In [16]:
# Implement unstructured pruning:
import torch.nn.utils.prune as prune

prune.l1_unstructured(model.features[27], name='weight', amount=0.9)

QuantConv2d(
  8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
  (weight_quant): weight_quantize_fn()
)

In [7]:
layer = model.features[27]

# 1. Register hook to mask quantized weights
def mask_quantized_weights(module, inputs, output):
    return output * layer.weight_mask

pruning_hook = layer.weight_quant.register_forward_hook(
    mask_quantized_weights
)

# 2. Perform forward pass
model.eval()

with torch.no_grad():
    x, _ = next(iter(testloader))
    output = model(x.to(device))

# 3. Verify quantized weights are pruned
mask = layer.weight_mask
weight_q = layer.weight_q.detach()

nonzero_pruned = (weight_q[mask == 0] != 0).sum().item()

print("Masked weights:", (mask == 0).sum().item())
print("Nonzero weights at masked positions:", nonzero_pruned)
print("Quantized weight sparsity:",
      (weight_q == 0).float().mean().item())

assert nonzero_pruned == 0

Masked weights: 518
Nonzero weights at masked positions: 0
Quantized weight sparsity: 0.8993055820465088


In [8]:
mask = model.features[27].weight_mask
sparsity_mask = (mask == 0).sum() / mask.nelement()
print("Squeezed layer sparsity: ", sparsity_mask)

Squeezed layer sparsity:  tensor(0.8993, device='cuda:0')


In [9]:
model.eval()
model.cuda()

correct = 0
test_loss = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)
print('\n Model Verified Test Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


 Model Verified Test Accuracy: 4095/10000 (41%)



In [10]:
fdir = f"final_model_unstruct/final_model_unstruct_checkpoints"
max_epochs = 10
best_prec = 0
for epoch in range(0, max_epochs):
    train(trainloader, model, criterion, optimizer, epoch)
        
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)
    
    #  # remember best precision and save checkpoint
    # is_best = prec > best_prec
    # best_prec = max(prec,best_prec)
    # print('best acc: {:1f}'.format(best_prec))
    # save_checkpoint({
    #     'epoch': epoch + 1,
    #     'state_dict': model.state_dict(),
    #     'best_prec': best_prec,
    #     'optimizer': optimizer.state_dict(),
    #     }, is_best, fdir, best_prec)

Epoch: [0][0/391]	Time 11.124 (11.124)	Data 10.852 (10.852)	Loss 1.4585 (1.4585)	Prec 59.375% (59.375%)
Epoch: [0][390/391]	Time 0.063 (0.053)	Data 0.000 (0.028)	Loss 0.3329 (0.6733)	Prec 83.750% (79.382%)
Validation starts
Test: [0/79]	Time 0.044 (0.044)	Loss 0.4791 (0.4791)	Prec 85.156% (85.156%)
 * Prec 84.550% 
best acc: 84.550000
Epoch: [1][0/391]	Time 0.085 (0.085)	Data 0.057 (0.057)	Loss 0.4051 (0.4051)	Prec 86.719% (86.719%)
Epoch: [1][390/391]	Time 0.025 (0.025)	Data 0.000 (0.000)	Loss 0.2523 (0.2377)	Prec 91.250% (92.354%)
Validation starts
Test: [0/79]	Time 0.046 (0.046)	Loss 0.3767 (0.3767)	Prec 89.844% (89.844%)
 * Prec 87.600% 
best acc: 87.600000
Epoch: [2][0/391]	Time 0.081 (0.081)	Data 0.053 (0.053)	Loss 0.2944 (0.2944)	Prec 92.188% (92.188%)
Epoch: [2][390/391]	Time 0.023 (0.024)	Data 0.000 (0.000)	Loss 0.1366 (0.1626)	Prec 95.000% (95.062%)
Validation starts
Test: [0/79]	Time 0.051 (0.051)	Loss 0.3155 (0.3155)	Prec 91.406% (91.406%)
 * Prec 89.000% 
best acc: 89.0000

In [11]:
layer = model.features[27]

# 1. Register hook to mask quantized weights
def mask_quantized_weights(module, inputs, output):
    return output * layer.weight_mask

# 2. Perform forward pass
model.eval()

with torch.no_grad():
    x, _ = next(iter(testloader))
    output = model(x.to(device))

# 3. Verify quantized weights are pruned
mask = layer.weight_mask
weight_q = layer.weight_q.detach()

nonzero_pruned = (weight_q[mask == 0] != 0).sum().item()

print("Masked weights:", (mask == 0).sum().item())
print("Nonzero weights at masked positions:", nonzero_pruned)
print("Quantized weight sparsity:",
      (weight_q == 0).float().mean().item())

assert nonzero_pruned == 0

Masked weights: 518
Nonzero weights at masked positions: 0
Quantized weight sparsity: 0.8993055820465088


In [12]:
mask = model.features[27].weight_mask
sparsity_mask = (mask == 0).sum() / mask.nelement()
print("Squeezed layer sparsity: ", sparsity_mask)

Squeezed layer sparsity:  tensor(0.8993, device='cuda:0')


In [13]:
model.eval()
model.cuda()

correct = 0
test_loss = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)
print('\n Model Verified Test Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


 Model Verified Test Accuracy: 9066/10000 (91%)



In [14]:
model.features[27].weight

tensor([[[[0.0000, 0.0000, -0.0000],
          [0.0000, 0.0000, 0.0623],
          [0.0000, -0.0000, 0.0603]],

         [[-0.0000, 0.0610, -0.0000],
          [0.0000, 0.0000, -0.0000],
          [0.0000, -0.0000, -0.0000]],

         [[0.0000, -0.0000, -0.0000],
          [0.0000, 0.0000, 0.0000],
          [0.0635, -0.0000, 0.0000]],

         [[0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, 0.0000]],

         [[-0.0000, 0.0000, -0.0000],
          [-0.0000, 0.0000, 0.0000],
          [-0.0000, 0.0607, 0.0605]],

         [[-0.0000, -0.0000, 0.0000],
          [0.0639, -0.0000, -0.0000],
          [-0.0000, -0.0000, 0.0604]],

         [[0.0000, -0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000],
          [0.0000, -0.0000, 0.0000]],

         [[-0.0000, 0.0636, 0.0000],
          [0.0000, 0.0000, 0.0000],
          [-0.0000, 0.0000, -0.0000]]],


        [[[-0.0000, -0.0000, 0.0627],
          [0.0000, 0.0000, 0.0000],
          [-